In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv pytorch-lightning


# MLFP05 Exercise 2.3 — Production Pipeline: ONNX Export + InferenceServer


**WHAT YOU'LL LEARN:**
After completing this file, you will be able to:
- Explain WHY you cannot deploy PyTorch directly to production
(dependency weight, GPU lock-in, no multi-language support) in
terms a non-technical manager can understand
- Export a trained CNN to ONNX format using kailash-ml's OnnxBridge
- Validate ONNX output matches PyTorch output (numerical fidelity)
- Serve predictions through kailash-ml's InferenceServer with
warm cache for low-latency inference
- Benchmark latency and throughput for production sizing
- Apply this to deploying the best model at a Singapore e-commerce
platform — latency targets, throughput planning, cost per inference


**PREREQUISITES: M5/ex_2/01_simple_cnn.py and 02_resnet_se.py (trained**
CNN models, ModelRegistry registration)

**ESTIMATED TIME: ~25 min**


**PHASES:**
1. THEORY  — Why ONNX, what it solves, deployment landscape
2. BUILD   — Train a model and export to ONNX via OnnxBridge
3. TRAIN   — (integrated with BUILD — we need a trained model to export)
4. VISUALISE — Latency distribution, PyTorch vs ONNX comparison
5. APPLY   — E-commerce deployment: latency, throughput, cost



In [ ]:
from __future__ import annotations

import asyncio
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from shared.mlfp05.ex_2 import (
    BATCH_SIZE,
    CLASS_NAMES,
    DEVICE,
    EPOCHS,
    N_CLASSES,
    count_parameters,
    create_visualizer,
    denormalise_cifar,
    init_engines,
    load_cifar10,
    register_model,
    train_model,
)
from kailash_ml.bridge.onnx_bridge import OnnxBridge
from kailash_ml.engines.inference_server import InferenceServer


## PHASE 1 — THEORY: Why You Cannot Deploy PyTorch Directly
Imagine you've built a perfect engine in your workshop. It runs
beautifully on the test bench. Now you need to install it in a car
that's already driving at 100 km/h. You can't bring the whole workshop
along — you need to ship JUST the engine, in a standard format that
any mechanic can install.
ONNX (Open Neural Network Exchange) is that standard format for ML.
WHY NOT DEPLOY PYTORCH DIRECTLY?
1. DEPENDENCY WEIGHT:
PyTorch + CUDA + all dependencies = 2-5 GB installed.
ONNX Runtime = ~50 MB. In a containerised deployment (Kubernetes),
this is the difference between 30-second cold starts and 2-second
cold starts.
2. GPU LOCK-IN:
PyTorch models trained on NVIDIA GPUs need NVIDIA GPUs to serve.
ONNX Runtime runs on CPU, NVIDIA GPU, AMD GPU, Apple Silicon,
Intel NPU, ARM — any hardware with an ONNX execution provider.
3. LANGUAGE BARRIER:
Your model was trained in Python. Production services might be in
Go, Java, C++, or Rust. ONNX Runtime has native bindings for all.
No Python GIL bottleneck in production.
4. OPTIMISATION:
ONNX Runtime applies graph-level optimisations: operator fusion,
constant folding, memory planning. A model that takes 15ms in
PyTorch often takes 5-8ms in ONNX Runtime — free speedup.
THE DEPLOYMENT PIPELINE:
Train (PyTorch) -> Export (ONNX) -> Validate -> Register (ModelRegistry)
-> Serve (InferenceServer / ONNX Runtime) -> Monitor (DriftMonitor)
InferenceServer wraps this pipeline: it loads models from the
ModelRegistry, caches them in memory (LRU), and serves predictions
via predict() and predict_batch(). For CNN models, the production
path uses ONNX Runtime directly for maximum performance.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: Why ONNX for Production Deployment")
print("=" * 70)
print(
    """
  PyTorch = workshop (great for building, too heavy for shipping)
  ONNX = standard engine format (any mechanic can install)

  Benefits:
    - 50 MB vs 5 GB dependency footprint
    - Hardware-agnostic (CPU, GPU, ARM, Apple Silicon)
    - Language-agnostic (Python, Go, Java, C++, Rust)
    - Free optimisation (graph fusion, constant folding)
    - 2-3x latency improvement typical

  Pipeline: Train -> Export -> Validate -> Register -> Serve -> Monitor
"""
)


## PHASE 2+3 — BUILD + TRAIN: Model for Export
We train a ResNetSE model (same as 02_resnet_se.py) and export it.
In production, you'd load the best model from ModelRegistry instead.


In [ ]:
print("=" * 70)
print("  PHASE 2+3 — BUILD + TRAIN: Prepare Model for Export")
print("=" * 70)


class ResBlock(nn.Module):
    """Residual block for the export model."""

    def __init__(self, channels: int):
        super().__init__()
        # TODO: Two Conv2d(channels, channels, 3, padding=1) with BatchNorm2d
        self.conv1 = ____
        self.bn1 = ____
        self.conv2 = ____
        self.bn2 = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: identity = x, apply conv->bn->relu->conv->bn, return relu(out + identity)
        identity = x
        out = ____
        out = ____
        return ____


class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for the export model."""

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        # TODO: SE MLP — Linear(channels, hidden), ReLU, Linear(hidden, channels), Sigmoid
        self.fc = nn.Sequential(
            ____,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        # TODO: squeeze -> excite -> scale
        s = ____
        w = ____
        return ____


class ResNetSE(nn.Module):
    """ResNet+SE model for ONNX export."""

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        # TODO: stem (Conv->BN->ReLU->MaxPool), block1, se1, block2, pool, fc
        self.stem = nn.Sequential(
            ____,
        )
        self.block1 = ____
        self.se1 = ____
        self.block2 = ____
        self.pool = ____
        self.fc = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: stem -> block1 -> se1 -> block2 -> pool -> flatten -> fc
        ____


# Load data and train
X_train, y_train, X_val, y_val, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()

# TODO: Train ResNetSE
#   resnet_se = ResNetSE()
#   resnet_losses, resnet_accs = train_model(resnet_se, "ResNetSE_for_export",
#       tracker, exp_name, train_loader, val_loader, epochs=EPOCHS)
print(f"\nTraining ResNetSE for ONNX export ({EPOCHS} epochs)...")
resnet_se = ____
resnet_losses, resnet_accs = ____

# Register in ModelRegistry
if has_registry:
    model_version = register_model(
        registry,
        "resnet_se_cifar10",
        resnet_se,
        resnet_losses[-1],
        resnet_accs[-1],
    )


### Checkpoint 1: Model trained ──────────────────────────────────────


In [ ]:
assert (
    resnet_accs[-1] > 0.4
), f"ResNetSE val accuracy {resnet_accs[-1]:.3f} too low for export"
print(f"\nModel ready: loss={resnet_losses[-1]:.4f}, val_acc={resnet_accs[-1]:.3f}")
print("--- Checkpoint 1 passed --- model trained and ready for export\n")


## ONNX Export via OnnxBridge


In [ ]:
print("=" * 70)
print("  ONNX EXPORT")
print("=" * 70)

bridge = OnnxBridge()
resnet_se.eval()
onnx_path = Path("ex_2_resnet_se.onnx")

# TODO: Export to ONNX
#   Try OnnxBridge first — bridge.export(model=resnet_se, framework="pytorch",
#       output_path=onnx_path, n_features=3*32*32)
#   If it fails for Conv2d, fall back to torch.onnx.export:
#       sample = torch.randn(1, 3, 32, 32)
#       torch.onnx.export(resnet_se, sample, onnx_path,
#           input_names=["input"], output_names=["logits"],
#           dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
#           opset_version=17, dynamo=False)
exported = False
try:
    result = ____
    success = getattr(result, "success", bool(result))
    print(f"  OnnxBridge.export success: {success}")
    exported = bool(success) and onnx_path.exists()
except Exception as exc:
    print(f"  OnnxBridge.export raised {type(exc).__name__}: {exc}")

if not exported:
    print("  Falling back to torch.onnx.export for Conv2D graph...")
    sample = torch.randn(1, 3, 32, 32)
    ____


### Checkpoint 2: ONNX file exists ──────────────────────────────────


In [ ]:
assert onnx_path.exists(), "ONNX file should exist after export"
onnx_size_kb = onnx_path.stat().st_size // 1024
print(f"  Wrote {onnx_path} ({onnx_size_kb} KB)")
print(
    "  The .onnx file contains the architecture AND learned weights.\n"
    "  Any ONNX Runtime can load and execute it without PyTorch."
)
print("\n--- Checkpoint 2 passed --- ONNX export complete\n")


## Numerical Validation: PyTorch vs ONNX Runtime


In [ ]:
print("=" * 70)
print("  NUMERICAL VALIDATION: PyTorch vs ONNX Runtime")
print("=" * 70)

ort_available = False
try:
    import onnxruntime as ort

    ort_available = True
except ImportError:
    print("  onnxruntime not installed -- skipping ONNX validation.")
    print("  Install with: pip install onnxruntime")

if ort_available:
    ort_session = ort.InferenceSession(str(onnx_path))
    input_name = ort_session.get_inputs()[0].name

    # Test on 100 validation images
    test_images = X_val[:100]
    test_np = test_images.numpy().astype(np.float32)

    # TODO: Compare PyTorch and ONNX Runtime predictions
    #   1. PyTorch: with torch.no_grad(): pt_logits = resnet_se(test_images).numpy()
    #   2. ONNX: ort_logits = ort_session.run(None, {input_name: test_np})[0]
    #   3. Compare: prediction_match = np.mean(pt_preds == ort_preds)
    #   4. max_logit_diff = np.max(np.abs(pt_logits - ort_logits))
    resnet_se.eval()
    with torch.no_grad():
        pt_logits = ____
    pt_preds = np.argmax(pt_logits, axis=-1)

    ort_logits = ____
    ort_preds = np.argmax(ort_logits, axis=-1)

    prediction_match = ____
    max_logit_diff = ____
    mean_logit_diff = np.mean(np.abs(pt_logits - ort_logits))

    print(
        f"  Prediction agreement: {prediction_match:.0%} ({int(prediction_match * 100)}/100)"
    )
    print(f"  Max logit difference:  {max_logit_diff:.6f}")
    print(f"  Mean logit difference: {mean_logit_diff:.6f}")

    assert prediction_match >= 0.99, (
        f"PyTorch vs ONNX prediction mismatch: {prediction_match:.0%} agreement. "
        "Export may have lost fidelity."
    )
    assert max_logit_diff < 0.01, (
        f"Max logit difference {max_logit_diff:.6f} too large. "
        "ONNX export should be numerically close to PyTorch."
    )
    print("  Numerical validation PASSED -- ONNX matches PyTorch")


## InferenceServer Setup


In [ ]:
print("\n" + "=" * 70)
print("  INFERENCE SERVER")
print("=" * 70)


async def setup_inference_server():
    """Demonstrate InferenceServer with ModelRegistry."""
    if not has_registry:
        print("  ModelRegistry not available -- skipping InferenceServer demo")
        return None

    # TODO: Create InferenceServer with registry and cache_size=5
    #   server = InferenceServer(registry=registry, cache_size=5)
    #   await server.warm_cache(["resnet_se_cifar10"])
    server = ____

    try:
        ____
        print("  InferenceServer cache warmed with resnet_se_cifar10")
    except Exception as e:
        print(f"  Cache warm note: {e}")

    return server


server = await setup_inference_server()

# Direct inference pipeline (the pattern InferenceServer wraps in production)
print("\n  Sample predictions via direct inference pipeline:")
resnet_se.eval()
indices = [0, 100, 500, 2000, 5000]
with torch.no_grad():
    sample_images = X_val[indices]
    logits = resnet_se(sample_images)
    probs = F.softmax(logits, dim=-1)
    preds = logits.argmax(dim=-1)

    for i, idx in enumerate(indices):
        pred_class = CLASS_NAMES[preds[i].item()]
        true_class = CLASS_NAMES[y_val[idx].item()]
        confidence = probs[i][preds[i]].item()
        status = "CORRECT" if preds[i].item() == y_val[idx].item() else "WRONG"
        print(
            f"    Sample {idx}: pred={pred_class:>10s} "
            f"(conf={confidence:.2f}) | true={true_class:>10s} [{status}]"
        )

# ONNX Runtime inference
if ort_available:
    print("\n  Same predictions via ONNX Runtime (production path):")
    sample_np = sample_images.numpy().astype(np.float32)
    ort_outputs = ort_session.run(None, {input_name: sample_np})
    ort_preds_sample = np.argmax(ort_outputs[0], axis=-1)

    for i, idx in enumerate(indices):
        pred_class = CLASS_NAMES[ort_preds_sample[i]]
        true_class = CLASS_NAMES[y_val[idx].item()]
        print(f"    ONNX Sample {idx}: pred={pred_class:>10s} | true={true_class:>10s}")


### Checkpoint 3: Inference pipeline works ───────────────────────────


In [ ]:
batch_acc_check = (preds == y_val[indices]).float().mean().item()
print(f"\n  Batch accuracy on samples: {batch_acc_check:.0%}")
print("--- Checkpoint 3 passed --- inference pipeline demonstrated\n")


## PHASE 4 — VISUALISE: Latency Benchmarks
Production ML is not just about accuracy — it is about SPEED and COST.
A model that takes 500ms per prediction cannot serve a website that
needs <100ms response time.


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: Latency Benchmarks")
print("=" * 70)

N_BENCHMARK = 100
single_image = X_val[:1]
batch_16 = X_val[:16]

# TODO: Benchmark PyTorch latency
#   1. Warmup: 10 forward passes on single_image
#   2. Time N_BENCHMARK single-image passes -> pt_latencies_single (ms)
#   3. Time N_BENCHMARK batch-16 passes -> pt_latencies_batch (ms)
#   Use time.perf_counter() for timing; multiply by 1000 for ms
resnet_se.eval()
pt_latencies_single = []
pt_latencies_batch = []

with torch.no_grad():
    # Warmup
    for _ in range(10):
        _ = resnet_se(single_image)

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        ____
        pt_latencies_single.append((time.perf_counter() - t0) * 1000)

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        ____
        pt_latencies_batch.append((time.perf_counter() - t0) * 1000)

# TODO: Benchmark ONNX Runtime latency (same pattern with ort_session.run)
ort_latencies_single = []
ort_latencies_batch = []

if ort_available:
    single_np = single_image.numpy().astype(np.float32)
    batch_np = batch_16.numpy().astype(np.float32)

    for _ in range(10):
        _ = ort_session.run(None, {input_name: single_np})

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        ____
        ort_latencies_single.append((time.perf_counter() - t0) * 1000)

    for _ in range(N_BENCHMARK):
        t0 = time.perf_counter()
        ____
        ort_latencies_batch.append((time.perf_counter() - t0) * 1000)

# Print benchmark results
pt_single_mean = np.mean(pt_latencies_single)
pt_single_p99 = np.percentile(pt_latencies_single, 99)
pt_batch_mean = np.mean(pt_latencies_batch)

print(f"\n  {'Metric':>30s} {'PyTorch':>12s}", end="")
if ort_available:
    print(f" {'ONNX RT':>12s} {'Speedup':>10s}")
else:
    print()
print("  " + "-" * 70)

metrics_to_print = [
    (
        "Single image (mean)",
        pt_single_mean,
        np.mean(ort_latencies_single) if ort_available else None,
    ),
    (
        "Single image (p99)",
        pt_single_p99,
        np.percentile(ort_latencies_single, 99) if ort_available else None,
    ),
    (
        "Batch of 16 (mean)",
        pt_batch_mean,
        np.mean(ort_latencies_batch) if ort_available else None,
    ),
    (
        "Per-image in batch",
        pt_batch_mean / 16,
        np.mean(ort_latencies_batch) / 16 if ort_available else None,
    ),
]

for label, pt_val, ort_val in metrics_to_print:
    line = f"  {label:>30s} {pt_val:>10.2f}ms"
    if ort_val is not None:
        speedup = pt_val / ort_val if ort_val > 0 else float("inf")
        line += f" {ort_val:>10.2f}ms {speedup:>9.1f}x"
    print(line)

# Throughput calculation
pt_throughput = 1000 / (pt_batch_mean / 16)
print(f"\n  PyTorch throughput: {pt_throughput:,.0f} images/second")
if ort_available:
    ort_throughput = 1000 / (np.mean(ort_latencies_batch) / 16)
    print(f"  ONNX RT throughput: {ort_throughput:,.0f} images/second")

# Visualise latency distributions
fig_latency, axes = plt.subplots(1, 2, figsize=(14, 5))
fig_latency.suptitle("Inference Latency: PyTorch vs ONNX Runtime", fontsize=14)

axes[0].hist(
    pt_latencies_single, bins=30, alpha=0.7, label="PyTorch", color="steelblue"
)
if ort_available:
    axes[0].hist(
        ort_latencies_single, bins=30, alpha=0.7, label="ONNX RT", color="coral"
    )
axes[0].set_xlabel("Latency (ms)")
axes[0].set_ylabel("Count")
axes[0].set_title("Single Image Inference")
axes[0].legend()
axes[0].axvline(
    np.mean(pt_latencies_single), color="steelblue", linestyle="--", alpha=0.5
)
if ort_available:
    axes[0].axvline(
        np.mean(ort_latencies_single), color="coral", linestyle="--", alpha=0.5
    )

axes[1].hist(pt_latencies_batch, bins=30, alpha=0.7, label="PyTorch", color="steelblue")
if ort_available:
    axes[1].hist(
        ort_latencies_batch, bins=30, alpha=0.7, label="ONNX RT", color="coral"
    )
axes[1].set_xlabel("Latency (ms)")
axes[1].set_ylabel("Count")
axes[1].set_title("Batch of 16 Inference")
axes[1].legend()

plt.tight_layout()
plt.savefig("ex_2_03_latency_benchmark.png", dpi=150, bbox_inches="tight")
plt.close(fig_latency)
print("\n  Saved: ex_2_03_latency_benchmark.png")

# Model size comparison
pytorch_size_mb = sum(p.numel() * p.element_size() for p in resnet_se.parameters()) / (
    1024 * 1024
)
print(f"\n  Model sizes:")
print(f"    PyTorch in-memory: {pytorch_size_mb:.2f} MB")
print(f"    ONNX file on disk: {onnx_size_kb / 1024:.2f} MB ({onnx_size_kb} KB)")


### Checkpoint 4: Benchmarks complete ────────────────────────────────


In [ ]:
import os

assert os.path.exists("ex_2_03_latency_benchmark.png"), "Latency plot missing"
assert pt_single_mean > 0, "PyTorch latency should be positive"
print("\n--- Checkpoint 4 passed --- latency benchmarks complete\n")


## PHASE 5 — APPLY: E-Commerce Deployment Planning
SCENARIO: You are deploying the product categorisation CNN from
01_simple_cnn.py to production at a Singapore e-commerce platform.
The engineering manager asks three questions:
1. "What hardware do we need?"
2. "How many images per second can we process?"
3. "What will it cost?"
You need concrete answers based on your benchmark data.


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: E-Commerce Deployment (Singapore Platform)")
print("=" * 70)

# Production parameters
DAILY_LISTINGS = 500_000
PEAK_MULTIPLIER = 3.0
LATENCY_SLA_MS = 100
MODEL_LATENCY_BUDGET_MS = 30

# TODO: Calculate deployment sizing from your benchmark data
#   1. avg_per_second = DAILY_LISTINGS / (24 * 3600)
#   2. peak_per_second = avg_per_second * PEAK_MULTIPLIER
#   3. Pick the runtime (ONNX if available, else PyTorch)
#   4. Calculate images_per_sec_per_replica from your latency data
#   5. With batching: batched_per_sec = 1000 / (batch_mean_ms / 16)
#   6. replicas_for_peak = ceil(peak_per_second / batched_per_sec), min 2
avg_per_second = ____
peak_per_second = ____

if ort_available:
    single_image_ms = np.mean(ort_latencies_single)
    runtime_name = "ONNX Runtime"
    images_per_sec_per_replica = 1000 / single_image_ms
else:
    single_image_ms = pt_single_mean
    runtime_name = "PyTorch"
    images_per_sec_per_replica = 1000 / single_image_ms

if ort_available:
    batched_per_sec = 1000 / (np.mean(ort_latencies_batch) / 16)
else:
    batched_per_sec = pt_throughput

replicas_for_peak = int(np.ceil(peak_per_second / batched_per_sec))
replicas_for_peak = max(replicas_for_peak, 2)

# TODO: Calculate costs (Singapore cloud pricing)
#   CPU_COST_PER_HOUR = 0.05  (c5.large equivalent)
#   GPU_COST_PER_HOUR = 0.90  (g4dn.xlarge equivalent)
#   cpu_replicas = max(replicas_for_peak * 2, 4)
#   cpu_monthly = cpu_replicas * CPU_COST_PER_HOUR * 24 * 30
#   gpu_monthly = replicas_for_peak * GPU_COST_PER_HOUR * 24 * 30
#   cost_per_inference_cpu = cpu_monthly / (DAILY_LISTINGS * 30)
#   cost_per_inference_gpu = gpu_monthly / (DAILY_LISTINGS * 30)
CPU_COST_PER_HOUR = 0.05
GPU_COST_PER_HOUR = 0.90

cpu_replicas = ____
cpu_monthly = ____
gpu_monthly = ____
cost_per_inference_cpu = ____
cost_per_inference_gpu = ____

print(
    f"""
  DEPLOYMENT SIZING (based on benchmark data):

  Traffic Profile:
    Daily listings:          {DAILY_LISTINGS:>10,}
    Average per second:      {avg_per_second:>10.1f}
    Peak per second (3x):    {peak_per_second:>10.1f}
    Latency SLA:             {LATENCY_SLA_MS:>10} ms (page load)
    Model latency budget:    {MODEL_LATENCY_BUDGET_MS:>10} ms

  Model Performance ({runtime_name}):
    Single image latency:    {single_image_ms:>10.2f} ms
    Batched throughput:      {batched_per_sec:>10.0f} images/sec/replica
    Meets latency budget:    {"YES" if single_image_ms < MODEL_LATENCY_BUDGET_MS else "NO":>10s}

  OPTION A — CPU Deployment (ONNX Runtime, recommended for this model):
    Replicas needed:         {cpu_replicas:>10}
    Instance type:           {"c5.large":>10s} (2 vCPU, 4 GB)
    Monthly cost:            ${cpu_monthly:>9,.0f}
    Cost per inference:      ${cost_per_inference_cpu:>9.6f}
    Pros: Simple, no GPU driver headaches, easy horizontal scaling
    Cons: Higher latency, more replicas needed

  OPTION B — GPU Deployment (ONNX Runtime + CUDA):
    Replicas needed:         {replicas_for_peak:>10}
    Instance type:           {"g4dn.xlarge":>10s} (T4 GPU, 4 vCPU, 16 GB)
    Monthly cost:            ${gpu_monthly:>9,.0f}
    Cost per inference:      ${cost_per_inference_gpu:>9.6f}
    Pros: Lower latency, fewer replicas, room for larger models
    Cons: GPU driver management, less elastic scaling

  RECOMMENDATION FOR THE ENGINEERING MANAGER:
    This model ({count_parameters(resnet_se):,} params, {onnx_size_kb} KB ONNX) is
    small enough for CPU deployment. GPU is overkill unless you plan to
    upgrade to a larger model (ResNet-50, EfficientNet) later.

    Start with Option A (CPU):
      - Deploy ONNX file to {cpu_replicas} CPU replicas behind a load balancer
      - Set auto-scaling: scale up at 70% CPU, scale down at 30%
      - Monitor with kailash-ml DriftMonitor for accuracy degradation
      - Budget: ${cpu_monthly:,.0f}/month (~${cpu_monthly * 12:,.0f}/year)

    Compared to manual categorisation ($10,000/day = $300,000/month):
      Savings: ${300000 - cpu_monthly:,.0f}/month = ${(300000 - cpu_monthly) * 12:,.0f}/year
"""
)


### Checkpoint 5: Apply section complete ─────────────────────────────


In [ ]:
assert replicas_for_peak >= 1, "Should need at least 1 replica"
assert cost_per_inference_cpu > 0, "Cost per inference should be positive"
print("--- Checkpoint 5 passed --- deployment planning complete\n")


## Clean up


In [ ]:
await conn.close()


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
    f"""
  THEORY:
  [x] Why PyTorch cannot go directly to production (2-5 GB footprint,
      GPU lock-in, Python GIL, no cross-language support)
  [x] ONNX as the universal exchange format (50 MB, any hardware,
      any language, free graph optimisations)
  [x] The deployment pipeline: Train -> Export -> Validate -> Register
      -> Serve -> Monitor

  BUILD + TRAIN:
  [x] Exported ResNetSE to ONNX via OnnxBridge / torch.onnx
  [x] Validated numerical fidelity: PyTorch vs ONNX predictions match
  [x] Set up InferenceServer with ModelRegistry and warm cache
  [x] ONNX file: {onnx_path} ({onnx_size_kb} KB)

  VISUALISE (the proof):
  [x] Latency distribution: PyTorch vs ONNX Runtime side-by-side
  [x] Single image: {pt_single_mean:.2f}ms (PyTorch){f" vs {np.mean(ort_latencies_single):.2f}ms (ONNX)" if ort_available else ""}
  [x] Throughput: {pt_throughput:,.0f} img/sec (PyTorch){f" vs {ort_throughput:,.0f} img/sec (ONNX)" if ort_available else ""}

  APPLY:
  [x] Singapore e-commerce deployment planning with concrete numbers
  [x] CPU vs GPU cost comparison for this model size
  [x] Recommendation: CPU deployment at ${cpu_monthly:,.0f}/month
  [x] Annual savings vs manual review: ${(300000 - cpu_monthly) * 12:,.0f}

  KEY INSIGHT: The model is just one artifact in the deployment pipeline.
  ONNX export, numerical validation, latency benchmarking, capacity
  planning, and cost analysis are what turn "it works on my laptop" into
  "it's running in production." An engineering manager does not care about
  your model's val_accuracy — they care about latency, throughput, cost,
  and reliability.

  Next: In 04_hyperparameter_study.py, you'll explore the accuracy-vs-cost
  tradeoff with systematic learning rate and augmentation experiments...
"""
)
